# NB4 – GPU: PhoBERT temporal hold-out 2019–2022 → 2023/2024/2025 (R2-C1)
**Runtime:** GPU **G4** · High-RAM  ·  **Thời gian ước tính:** khoảng 1,5 giờ

Notebook tự chứa: code được nhúng sẵn (các ô `%%writefile`), không cần upload file.
Có thể chạy **song song** với các notebook khác. Chọn *Runtime → Run all*.
Kết quả được chép vào `MyDrive/PLOS_revision/NB4_GPU_G3_temporal/`.

Nối các bộ dữ liệu để lấy năm đăng, rồi fine-tune PhoBERT trên 2019–2022 và đánh giá riêng từng năm 2023, 2024, 2025.

In [ ]:
# 1) Drive (kết quả được chép vào đây) + 2) đăng nhập Hugging Face (tài khoản BaoNhan)
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
RESULT_DIR = "/content/drive/MyDrive/PLOS_revision/NB4_GPU_G3_temporal"
os.makedirs(RESULT_DIR, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # Colab secret tên HF_TOKEN (nếu có)
except Exception:
    from huggingface_hub import login
    login()                                                 # nếu chưa có secret: dán token vào ô hiện ra
from huggingface_hub import whoami
try:
    print("Đã đăng nhập Hugging Face:", whoami()["name"])
except Exception:
    raise SystemExit("CHƯA ĐĂNG NHẬP Hugging Face. Bấm biểu tượng chìa khóa (Secrets) bên trái, "
                     "bật 'Notebook access' cho HF_TOKEN, rồi chạy lại ô này.")
!pip -q install datasets pyarrow scikit-learn

## Code (nhúng sẵn – không cần sửa)

In [ ]:
%%writefile common.py
"""Shared configuration for the PLOS ONE revision analyses (PONE-D-26-38656).

Every constant below is taken from the manuscript or from the Hugging Face
repositories listed by step0_inspect.py. Nothing here is estimated.
"""
from pathlib import Path

SEEDS = [42, 22, 202]
FIG_SEED_ORDER = [22, 42, 202]          # panel order of the original Fig 5
MAX_LEN = 256                           # model-specific token budget
N_CLASSES = 10

# Model display name -> (HF repo, short key used in folder/file names)
MODELS = {
    "WikiBERT":   ("BaoNhan/WikiBERT_Vietnamese_News_Classification", "wikibert"),
    "PhoBERT":    ("BaoNhan/PhoBERT_Vietnamese_News_Classification", "phobert"),
    "mBERT":      ("BaoNhan/mBERT_Vietnamese_News_Classification", "mbert"),
    "DistilBERT": ("BaoNhan/DistilBERT_Vietnamese_News_Classification", "distilbert"),
    "XLM-R":      ("BaoNhan/XLMR_Vietnamese_News_Classification", "xlmr"),
    "CafeBERT":   (None, "cafebert"),   # fine-tuned CafeBERT runs are not on the Hub
    "viBERT":     ("BaoNhan/viBERT_Vietnamese_News_Classification", "vibert"),
    "vELECTRA":   ("BaoNhan/vELECTRA_Vietnamese_News_Classification", "velectra"),
    "ViSoBERT":   ("BaoNhan/ViSoBERT_Vietnamese_News_Classification", "visobert"),
}
MODEL_ORDER = list(MODELS)  # same order as Table 2
# Public pre-trained checkpoints that were fine-tuned (verified to exist on the Hub;
# step4 also checks their sizes against the released fine-tuned configs).
BASE_CHECKPOINTS = {
    "WikiBERT":   "TurkuNLP/wikibert-base-vi-cased",
    "PhoBERT":    "vinai/phobert-base",
    "mBERT":      "google-bert/bert-base-multilingual-cased",
    "DistilBERT": "distilbert/distilbert-base-multilingual-cased",
    "XLM-R":      "FacebookAI/xlm-roberta-base",
    "CafeBERT":   "uitnlp/CafeBERT",
    "viBERT":     "FPTAI/vibert-base-cased",
    "vELECTRA":   "FPTAI/velectra-base-discriminator-cased",
    "ViSoBERT":   "uitnlp/visobert",
}
# Models whose seed-specific TCC was not retained: Fig 5 uses the Table 2 mean TCC
# for every seed of these models and the caption states this explicitly.
MEAN_TCC_FALLBACK = ["PhoBERT", "CafeBERT"]

# Category indices used by all prediction files (manuscript, Results section)
LABEL_NAMES = ["Education", "Entertainment", "Business", "Law", "Health",
               "World", "Sports", "Politics", "Automotive", "Lifestyle"]
# Test-set row totals per index, as reported in the manuscript
TEST_COUNTS = [6090, 9542, 12350, 6992, 9766, 13673, 12423, 13490, 4844, 5500]
N_TEST = 94670

# Table 2 of the submitted manuscript: mean (population SD) over seeds
TABLE2 = {  # loss, acc, P, R, F1, AUC, TCC
    "WikiBERT":   [0.20330, 0.93016, 0.92377, 0.92264, 0.92302, 0.99635, 3077.08],
    "PhoBERT":    [0.14761, 0.94779, 0.94124, 0.94375, 0.94241, 0.99794, 3360.89],
    "mBERT":      [0.16421, 0.94270, 0.93707, 0.93660, 0.93674, 0.99750, 3181.59],
    "DistilBERT": [0.17685, 0.93793, 0.93145, 0.93213, 0.93167, 0.99713, 1554.86],
    "XLM-R":      [0.15650, 0.94488, 0.93907, 0.94011, 0.93953, 0.99770, 3099.05],
    "CafeBERT":   [0.14729, 0.94916, 0.94391, 0.94445, 0.94413, 0.99800, 9335.51],
    "viBERT":     [0.20287, 0.92903, 0.92226, 0.92185, 0.92190, 0.99629, 3065.10],
    "vELECTRA":   [0.18201, 0.93791, 0.93174, 0.93189, 0.93161, 0.99710, 3192.59],
    "ViSoBERT":   [0.21663, 0.92617, 0.91819, 0.92075, 0.91934, 0.99604, 3201.82],
}
TABLE2_TCC_SD = {"WikiBERT": 3.46492, "PhoBERT": 4.53214, "mBERT": 1.24008,
                 "DistilBERT": 0.99322, "XLM-R": 1.41930, "CafeBERT": 148.46973,
                 "viBERT": 1.53178, "vELECTRA": 1.57443, "ViSoBERT": 0.44646}

COLORS = {"WikiBERT": "#4C78A8", "PhoBERT": "#F58518", "mBERT": "#54A24B",
          "DistilBERT": "#E45756", "XLM-R": "#B279A2", "CafeBERT": "#72B7B2",
          "viBERT": "#FF9DA6", "vELECTRA": "#9D755D", "ViSoBERT": "#BAB0AC"}

DATA = Path("revision_data")      # created by step1
OUT = Path("revision_results")    # created by step2/step3


def pred_path(model, seed):
    return DATA / "preds" / f"{model}_seed{seed}.npz"

In [ ]:
%%writefile step1_collect.py
"""STEP 1 - Download the released predictions/metrics and assemble them.

Runs on CPU. Requires `huggingface_hub`, `datasets`, `pandas`, `numpy`.
Output folder: revision_data/
  preds/<Model>_seed<s>.npz   true, pred, prob (aligned row by row)
  test_text.pkl           row, id, content of the 94,670 test articles
  metrics/<Model>.json        raw per-seed metric JSON(s) as released
  dataset_meta.pkl        label/url/source of BaoNhan/Vietnam_News_3_Sources
  step1_report.txt            every check performed and its outcome

The script STOPS with an explanation if any consistency check fails.
"""
import json
import sys

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

from common import (DATA, MODELS, N_TEST, SEEDS, TEST_COUNTS,
                    pred_path)

api = HfApi()
(DATA / "preds").mkdir(parents=True, exist_ok=True)
(DATA / "metrics").mkdir(parents=True, exist_ok=True)
report = []


def log(msg):
    print(msg)
    report.append(msg)


def fail(msg):
    log("STOP: " + msg)
    (DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


# ---------------------------------------------------------------- locate files
locations = {}  # (model, seed) -> (repo, csv path)
metric_files = {}  # model -> list of (repo, path)
import os
ONLY = [m.strip() for m in os.environ.get("ONLY_MODELS", "").split(",") if m.strip()]
for model, (repo, key) in MODELS.items():
    if repo is None or (ONLY and model not in ONLY):
        continue
    files = api.list_repo_files(repo)
    for s in SEEDS:
        p = f"results_{key}/predictions_seed_{s}.csv"
        if p not in files:
            fail(f"{repo} has no {p}")
        locations[(model, s)] = (repo, p)
    metric_files[model] = [(repo, f) for f in files
                           if f.startswith(f"results_{key}/") and f.endswith(".json")]

metric_files["CafeBERT"] = []
log("CafeBERT: fine-tuned predictions are not released on the Hub - skipped")

# ---------------------------------------------------------------- predictions
PROB_COLS = [f"prob_class_{k}" for k in range(10)]
ref_true, ref_id, text_saved = None, None, False
for (model, s), (repo, path) in sorted(locations.items()):
    local = hf_hub_download(repo, path)
    header = pd.read_csv(local, nrows=0).columns.tolist()
    use = [c for c in header if c != "content"]
    df = pd.read_csv(local, usecols=use)
    if len(df) != N_TEST:
        fail(f"{model} seed {s}: {len(df)} rows, expected {N_TEST}")
    for c in ["true_label_0_9", "predicted_label_0_9"] + PROB_COLS:
        if c not in df.columns:
            fail(f"{model} seed {s}: column {c} missing; columns = {header}")
    true = df["true_label_0_9"].to_numpy(np.int8)
    pred = df["predicted_label_0_9"].to_numpy(np.int8)
    prob = df[PROB_COLS].to_numpy(np.float32)
    if ref_true is None:
        ref_true = true
        counts = np.bincount(true, minlength=10).tolist()
        if counts != TEST_COUNTS:
            fail(f"class counts {counts} differ from manuscript {TEST_COUNTS}")
        log(f"class counts match the manuscript: {counts}")
    elif not np.array_equal(true, ref_true):
        fail(f"{model} seed {s}: true-label order differs from the reference file")
    if "id" in df.columns:
        ids = df["id"].to_numpy()
        if ref_id is None:
            ref_id = ids
        elif not np.array_equal(ids, ref_id):
            fail(f"{model} seed {s}: id order differs from the reference file")
        if not text_saved:
            txt = pd.read_csv(local, usecols=["id", "content"])
            txt.insert(0, "row", np.arange(len(txt)))
            txt.to_pickle(DATA / "test_text.pkl")
            text_saved = True
            log(f"test text saved from {repo}/{path}")
    np.savez_compressed(pred_path(model, s), true=true, pred=pred, prob=prob)
    log(f"{model:10s} seed {s:3d}: OK  acc={np.mean(true == pred):.5f}  "
        f"id column={'id' in df.columns}")

missing = [(m, s) for m in MODELS for s in SEEDS if (m, s) not in locations and (not ONLY or m in ONLY)]
log(f"prediction files missing: {missing if missing else 'none'}")

# ---------------------------------------------------------------- metrics json
for model, lst in metric_files.items():
    merged = {}
    for repo, f in lst:
        merged[f"{repo}/{f}"] = json.load(open(hf_hub_download(repo, f), encoding="utf-8"))
    (DATA / "metrics" / f"{model}.json").write_text(
        json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    log(f"{model}: {len(lst)} metric JSON file(s) saved")

# ---------------------------------------------------------------- dataset meta
try:
    from datasets import load_dataset
    ds = load_dataset("BaoNhan/Vietnam_News_3_Sources")["train"].to_pandas()
    ds.insert(0, "row", np.arange(len(ds)))
    ds.to_pickle(DATA / "dataset_meta.pkl")
    log(f"dataset metadata saved: {len(ds)} rows, columns {list(ds.columns)}")
    if ref_id is not None:
        log(f"test id examples: {ref_id[:5].tolist()}  dtype={ref_id.dtype}")
except Exception as e:  # noqa: BLE001
    log(f"dataset metadata NOT saved: {e!r}")

(DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
print("\nDone. Send revision_data/step1_report.txt.")

In [ ]:
%%writefile step1b_index.py
"""STEP 1b - Link the three private datasets row by row (CPU, high-RAM runtime).

  BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split  label(int), content  train/dev/test
  BaoNhan/Vietnamese_News_PhoBERT_Segmented        same rows, word-segmented content
  BaoNhan/Vietnam_News_3_Sources_Plus              url, label(str), content, source,
                                                   day, month, year (one split)

Output revision_data/corpus_index.pkl - one row per Cleaned article:
  split, pos, label (0-9 index used by the prediction files), source, year, url,
  plus_row (row in the Plus dataset, -1 if not found), h_exact, h_norm (content hashes)
and revision_data/step1b_report.txt. Every link is verified, nothing is assumed:
  * Cleaned and Segmented must have identical labels at every (split, pos)
  * the Cleaned test labels must equal the prediction-file labels (offset detected)
  * Plus rows are matched to Cleaned rows by an exact hash of the content
    (a whitespace-normalised hash is used only for rows without an exact match)
  * the category names implied by the match are checked against the manuscript
"""
import hashlib
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

from common import DATA, LABEL_NAMES, MODEL_ORDER, SEEDS, pred_path

CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
SEG = "BaoNhan/Vietnamese_News_PhoBERT_Segmented"
PLUS = "BaoNhan/Vietnam_News_3_Sources_Plus"
SPLIT_SIZES = {"train": 757356, "dev": 94670, "test": 94670}
report = []
WS = re.compile(r"\s+")


def log(msg=""):
    print(msg)
    report.append(str(msg))


def stop(msg):
    log("STOP: " + msg)
    (DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


def h_exact(t):
    return hashlib.blake2b((t or "").encode("utf-8"), digest_size=8).hexdigest()


def h_norm(t):
    return h_exact(WS.sub(" ", t or "").strip())


def read_repo(repo, prefix, columns):
    """Stream a split's parquet shards; content is reduced to hashes immediately."""
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.startswith(f"data/{prefix}-") and f.endswith(".parquet"))
    out = []
    for f in files:
        pf = pq.ParquetFile(hf_hub_download(repo, f, repo_type="dataset"))
        for b in pf.iter_batches(batch_size=20000, columns=columns):
            d = b.to_pandas()
            if "content" in d:
                d["h_exact"] = d["content"].map(h_exact)
                d["h_norm"] = d["content"].map(h_norm)
                d = d.drop(columns="content")
            out.append(d)
    return pd.concat(out, ignore_index=True)


# -------------------------------------------------------- Cleaned and Segmented
parts = []
for split, n in SPLIT_SIZES.items():
    c = read_repo(CLEAN, split, ["label", "content"])
    s = read_repo(SEG, split, ["label"])
    if len(c) != n or len(s) != n:
        stop(f"{split}: Cleaned {len(c)} / Segmented {len(s)} rows, expected {n}")
    if not np.array_equal(c["label"].to_numpy(), s["label"].to_numpy()):
        stop(f"{split}: Cleaned and Segmented labels differ - rows are not aligned")
    c.insert(0, "split", split)
    c.insert(1, "pos", np.arange(n))
    parts.append(c)
    log(f"{split}: {n} rows, Cleaned/Segmented labels identical at every position; "
        f"label range {c['label'].min()}..{c['label'].max()}")
idx = pd.concat(parts, ignore_index=True)

# -------------------------------------------------------- label offset vs predictions
ref = next((pred_path(m, s) for m in MODEL_ORDER for s in SEEDS if pred_path(m, s).exists()), None)
if ref is None:
    stop("no prediction file found - run step1_collect.py first")
true = np.load(ref)["true"].astype(int)
test_lab = idx.loc[idx.split == "test", "label"].to_numpy()
offset = next((k for k in (0, 1) if np.array_equal(test_lab - k, true)), None)
if offset is None:
    stop("Cleaned test labels do not equal the prediction-file labels (offset 0 or 1)")
idx["label"] = idx["label"] - offset
log(f"Cleaned test split == prediction-file order; dataset label = index + {offset}")

# -------------------------------------------------------- Plus metadata
plus = read_repo(PLUS, "train", ["url", "label", "content", "source", "year"])
plus.insert(0, "plus_row", np.arange(len(plus)))
log(f"Plus: {len(plus)} rows; sources {plus.source.value_counts().to_dict()}")
for key in ("h_exact", "h_norm"):
    d = plus[key].duplicated(keep=False).sum()
    log(f"Plus rows sharing a {key} with another row: {d}")

m = idx.merge(plus.drop_duplicates("h_exact")[["h_exact", "plus_row"]], on="h_exact", how="left")
miss = m.plus_row.isna()
if miss.any():
    alt = plus.drop_duplicates("h_norm")[["h_norm", "plus_row"]].rename(columns={"plus_row": "pr2"})
    m = m.merge(alt, on="h_norm", how="left")
    m.loc[miss, "plus_row"] = m.loc[miss, "pr2"]
    m = m.drop(columns="pr2")
m["plus_row"] = m["plus_row"].fillna(-1).astype(int)
log(f"Cleaned rows matched to Plus: {(m.plus_row >= 0).mean():.4%} "
    f"(exact {(~miss).mean():.4%})")
meta = plus.set_index("plus_row")
ok = m.plus_row >= 0
for col in ("url", "source", "year"):
    m.loc[ok, col] = meta.loc[m.loc[ok, "plus_row"], col].to_numpy()
m.loc[ok, "plus_label"] = meta.loc[m.loc[ok, "plus_row"], "label"].to_numpy()

# -------------------------------------------------------- category-name check
ct = pd.crosstab(m.loc[ok, "label"], m.loc[ok, "plus_label"])
implied = ct.idxmax(axis=1).reindex(range(10)).tolist()
purity = (ct.max(axis=1) / ct.sum(axis=1)).round(4).tolist()
log(f"category name implied by Plus for index 0..9: {implied}")
log(f"share of rows agreeing with that name: {purity}")
log(f"manuscript names for index 0..9:          {LABEL_NAMES}")
log("NAMES AGREE" if implied == LABEL_NAMES else "NAMES DIFFER - report this before using them")
for split in SPLIT_SIZES:
    s = m[m.split == split]
    log(f"{split}: years {s.year.value_counts().sort_index().to_dict()}")

m.to_pickle(DATA / "corpus_index.pkl")
(DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
print("Done.")

In [ ]:
%%writefile step4_gpu.py
"""STEP 4 - The only GPU work for the revision (Colab G4, about 3-3.5 GPU-hours).

  G1  CafeBERT inference (seeds 42 and 22, checkpoints in Drive/CafeBERT_Results)
      -> article-level predictions -> PAIRED McNemar test and paired bootstrap of
         macro-F1 against PhoBERT-base (Reviewer #1, Concern 5).            ~10 min
  G2  PhoBERT-base, seed 42: head-only truncation (baseline, rerun on G4) versus
      head+tail truncation (first + last tokens within the same 256 budget)
      (Reviewer #2, Concern 3).                                              ~2 h
  G3  PhoBERT-base, seed 42: temporal hold-out - train/dev on 2019-2022 articles,
      test separately on all 2023, 2024 and 2025 articles (Reviewer #2, Concern 1).
                                                                             ~45 min
Every run is saved to Google Drive and skipped when re-launched (resumable).
Requires the outputs of step1_collect.py and step1b_index.py in revision_data/.

    from google.colab import drive; drive.mount('/content/drive')
    !python step4_gpu.py            # all stages
    !python step4_gpu.py G1         # one stage
"""
import glob
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from scipy.stats import binomtest
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

from common import DATA, SEEDS, pred_path

# ----------------------------------------------------------------- configuration
OUT = Path(os.environ.get("STEP4_OUT", "/content/drive/MyDrive/PLOS_revision_step4"))
OUT.mkdir(parents=True, exist_ok=True)
SEG = "BaoNhan/Vietnamese_News_PhoBERT_Segmented"
CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
PHOBERT = "vinai/phobert-base"
MAX_LEN, EPOCHS, WARMUP = 256, 3, 0.06
# Training batch: 256 in the manuscript (used by G3). G2 may be run with a larger
# batch (TRAIN_BATCH=512) for speed; both G2 runs then use the same value, so the
# head vs head+tail comparison stays like-for-like. EVAL_BATCH only affects speed.
BATCH = int(os.environ.get("TRAIN_BATCH", 256))
EVAL_BATCH = int(os.environ.get("EVAL_BATCH", 1024))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
# Learning rate for the PhoBERT reruns. The released PhoBERT trainer_state.json logs
# 2.1045e-06 at step 8000 of 8877 with 6% warm-up, which corresponds to a peak of 2e-5.
# Using the same value lets the G4 baseline reproduce the Table 2 PhoBERT result.
PHOBERT_LR = float(os.environ.get("PHOBERT_LR", 2e-5))
SEED = 42
CAFE_EXPECTED_CORRECT = {42: 89885, 22: 89848}   # from all_seeds_metrics_cafebert.json
STAGES = sys.argv[1:] or ["G1", "G2", "G3", "SUMMARY"]
dev = "cuda"
log_lines = []


def log(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))
    (OUT / "step4_report.txt").write_text("\n".join(log_lines), encoding="utf-8")


def metrics(y, prob):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return dict(n=int(len(y)), accuracy=float(np.mean(pred == y)), macro_precision=float(p),
                macro_recall=float(r), macro_f1=float(f),
                macro_auc=float(roc_auc_score(y, prob, multi_class="ovr", average="macro")))


def reference_true():
    return np.load(pred_path("PhoBERT", 42))["true"].astype(int)


def load_split(repo, split):
    ds = load_dataset(repo, split=split)
    return ds


def label_offset(test_labels):
    true = reference_true()
    for k in (0, 1):
        if np.array_equal(np.asarray(test_labels) - k, true):
            return k
    sys.exit("dataset test labels are not in prediction-file order")


NPROC = os.cpu_count()


def tokenize_all(tok, texts, mode, labels=None):
    """Tokenize in parallel with NPROC worker processes (the PhoBERT tokenizer is slow)."""
    data = {"text": list(texts)}
    if labels is not None:
        data["labels"] = list(labels)
    return Dataset.from_dict(data).map(
        lambda b: {"input_ids": encode(tok, b["text"], mode)}, batched=True, batch_size=1000,
        num_proc=NPROC, remove_columns=["text"])


@torch.no_grad()
def predict(model, tok, texts, mode="head"):
    model.eval().to(dev)
    ids = tokenize_all(tok, texts, mode)["input_ids"]
    order = np.argsort([len(x) for x in ids])          # length-sorted batches: less padding
    probs = np.zeros((len(ids), model.config.num_labels), dtype=np.float32)
    for i in range(0, len(ids), EVAL_BATCH):
        idx = order[i:i + EVAL_BATCH]
        batch = tok.pad({"input_ids": [ids[j] for j in idx]}, return_tensors="pt").to(dev)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**batch).logits
        probs[idx] = torch.softmax(logits.float(), -1).cpu().numpy()
    return probs


def encode(tok, texts, mode):
    if mode == "head":
        return tok(texts, truncation=True, max_length=MAX_LEN)["input_ids"]
    budget = MAX_LEN - tok.num_special_tokens_to_add(pair=False)
    head = budget // 2
    out = []
    for ids in tok(texts, add_special_tokens=False, truncation=False)["input_ids"]:
        if len(ids) > budget:
            ids = ids[:head] + ids[-(budget - head):]
        out.append(tok.build_inputs_with_special_tokens(ids))
    return out


def finetune(name, train_texts, train_y, dev_texts, dev_y, mode):
    """Fine-tune PhoBERT with the manuscript protocol; returns (model, tok, seconds)."""
    tok = AutoTokenizer.from_pretrained(PHOBERT)

    tr = tokenize_all(tok, train_texts, mode, train_y)
    dv = tokenize_all(tok, dev_texts, mode, dev_y)
    log(f"{name}: tokenized {len(tr)} train / {len(dv)} dev articles with {NPROC} processes")
    model = AutoModelForSequenceClassification.from_pretrained(PHOBERT, num_labels=10)
    steps = math.ceil(len(tr) / BATCH) * EPOCHS

    def compute(ev):
        logits, y = ev
        e = np.exp(logits - logits.max(1, keepdims=True))
        return {k: v for k, v in metrics(y, e / e.sum(1, keepdims=True)).items() if k != "n"}
    args = TrainingArguments(
        output_dir=f"/content/step4_ckpt/{name}",  # local disk, not Drive
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=EVAL_BATCH, num_train_epochs=EPOCHS, learning_rate=PHOBERT_LR,
        optim="adamw_torch_fused", dataloader_pin_memory=True,
        warmup_steps=math.ceil(WARMUP * steps), lr_scheduler_type="linear", bf16=True,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True, save_total_limit=1,
        seed=SEED, logging_steps=500, report_to="none", disable_tqdm=True,
        dataloader_num_workers=min(8, os.cpu_count()))
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=dv,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute)
    t0 = time.time()
    trainer.train()
    return trainer.model, tok, time.time() - t0


def save_run(name, results, extra=None):
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    json.dump(dict(results=results, **(extra or {})), open(d / "metrics.json", "w"), indent=2)


def done(name):
    return (OUT / name / "metrics.json").exists()


# ================================================================= G1 CafeBERT
def stage_g1():
    test = load_split(CLEAN, "test")
    off = label_offset(test["label"])
    y = np.asarray(test["label"]) - off
    texts = test["content"]
    roots = [os.environ["CAFE_DIR"]] if os.environ.get("CAFE_DIR") else [
        p for depth in ("*", "*/*", "*/*/*") for name in ("CafeBERT_Results", "CafeBERT_RESULTS")
        for p in glob.glob(f"/content/drive/{depth}/{name}")]
    if not roots:
        sys.exit("CafeBERT_Results not found under /content/drive (mount Drive / add a shortcut)")
    for s in (42, 22):
        name = f"G1_cafebert_seed{s}"
        if done(name):
            continue
        ck = sorted(glob.glob(f"{roots[0]}/cafebert_checkpoints_seed_{s}/checkpoint-*"))
        if len(ck) != 1:
            log(f"G1 seed {s}: expected one checkpoint folder, found {ck}")
            continue
        cfg_dir = ck[0]
        if not os.path.exists(f"{ck[0]}/config.json"):
            # the seed-42 folder holds only the weights and tokenizer.json; the seed-22
            # folder holds the config of the same XLMRobertaForSequenceClassification
            cfg_dir = sorted(glob.glob(f"{roots[0]}/cafebert_checkpoints_seed_*/checkpoint-*/config.json"))[0]
            cfg_dir = os.path.dirname(cfg_dir)
            log(f"G1 seed {s}: config.json missing, using {cfg_dir}/config.json")
        from transformers import AutoConfig
        model = AutoModelForSequenceClassification.from_pretrained(
            ck[0], config=AutoConfig.from_pretrained(cfg_dir))
        # The seed-42 folder has tokenizer.json but no tokenizer_config.json, so the
        # special tokens (<pad>, <s>, </s>) are not registered. Use a folder that has
        # both files (same CafeBERT tokenizer), else the public uitnlp/CafeBERT one.
        tok_dirs = [d for d in [ck[0]] + sorted(glob.glob(f"{roots[0]}/cafebert_checkpoints_seed_*/checkpoint-*"))
                    if os.path.exists(f"{d}/tokenizer_config.json")]
        tok = AutoTokenizer.from_pretrained(tok_dirs[0]) if tok_dirs else None
        if tok is None or tok.pad_token is None:
            tok = AutoTokenizer.from_pretrained("uitnlp/CafeBERT")
            log(f"G1 seed {s}: tokenizer loaded from uitnlp/CafeBERT")
        else:
            log(f"G1 seed {s}: tokenizer loaded from {tok_dirs[0]}")
        if tok.pad_token is None or tok.pad_token_id != model.config.pad_token_id:
            sys.exit(f"G1 seed {s}: pad token {tok.pad_token!r}/{tok.pad_token_id} does not match "
                     f"model pad_token_id {model.config.pad_token_id}")
        prob = predict(model, tok, texts)
        correct = int(np.sum(prob.argmax(1) == y))
        log(f"G1 seed {s}: {correct} correct (released JSON: {CAFE_EXPECTED_CORRECT[s]}, "
            f"difference {correct - CAFE_EXPECTED_CORRECT[s]})")
        np.savez_compressed(OUT / f"{name}_pred.npz", true=y, pred=prob.argmax(1), prob=prob)
        save_run(name, metrics(y, prob), dict(correct=correct, expected=CAFE_EXPECTED_CORRECT[s]))
        del model
        torch.cuda.empty_cache()


# ================================================================= G2 head vs head+tail
def stage_g2():
    tr, dv, te = (load_split(SEG, s) for s in ("train", "dev", "test"))
    off = label_offset(te["label"])
    for mode in ("head", "headtail"):
        name = f"G2_phobert_seed{SEED}_{mode}_bs{BATCH}"
        legacy = f"G2_phobert_seed{SEED}_{mode}"  # earlier runs (batch 256) had no suffix
        if done(name) or (BATCH == 256 and done(legacy)):
            log(f"G2 {mode} (batch {BATCH}): already done, skipped")
            continue
        model, tok, secs = finetune(name, tr["content"], [l - off for l in tr["label"]],
                                    dv["content"], [l - off for l in dv["label"]], mode)
        y = np.asarray(te["label"]) - off
        prob = predict(model, tok, te["content"], mode)
        np.savez_compressed(OUT / f"{name}_pred.npz", true=y, pred=prob.argmax(1), prob=prob)
        save_run(name, metrics(y, prob), dict(train_seconds=secs, lr=PHOBERT_LR, train_batch=BATCH))
        log(f"G2 {mode} (batch {BATCH}, {secs/60:.1f} min): {metrics(y, prob)}")
        del model
        torch.cuda.empty_cache()


# ================================================================= G3 temporal
def stage_g3():
    name = f"G3_phobert_seed{SEED}_temporal"
    if done(name):
        return
    ci = pd.read_pickle(DATA / "corpus_index.pkl")
    sp = {s: load_split(SEG, s) for s in ("train", "dev", "test")}
    off = label_offset(sp["test"]["label"])
    ci["year"] = pd.to_numeric(ci["year"], errors="coerce")
    log(f"G3: rows without a publication year: {int(ci.year.isna().sum())} (excluded)")

    def rows(mask):
        sub = ci[mask]
        texts, ys = [], []
        for s in ("train", "dev", "test"):
            pos = sub.loc[sub.split == s, "pos"].to_numpy()
            if len(pos):
                part = sp[s].select(pos)
                texts += part["content"]
                ys += [l - off for l in part["label"]]
        return texts, np.asarray(ys)
    early = ci.year.between(2019, 2022)
    trX, trY = rows(early & (ci.split == "train"))
    dvX, dvY = rows(early & (ci.split == "dev"))
    log(f"G3: train {len(trY)} / dev {len(dvY)} articles from 2019-2022")
    model, tok, secs = finetune(name, trX, trY.tolist(), dvX, dvY.tolist(), "head")
    res = {}
    for label, mask in [("2019-2022 (original test split)", early & (ci.split == "test")),
                        ("2023", ci.year == 2023), ("2024", ci.year == 2024), ("2025", ci.year == 2025)]:
        X, Y = rows(mask)
        res[label] = metrics(Y, predict(model, tok, X))
        log(f"G3 {label}: {res[label]}")
    save_run(name, res, dict(train_seconds=secs, lr=PHOBERT_LR, train_batch=BATCH, n_train=len(trY), n_dev=len(dvY)))


# ================================================================= summary
def stage_summary():
    out = []
    pho = {s: np.load(pred_path("PhoBERT", s)) for s in SEEDS}
    rng = np.random.default_rng(20261106)
    for s in (42, 22):
        f = OUT / f"G1_cafebert_seed{s}_pred.npz"
        if not f.exists():
            continue
        c, p = np.load(f), pho[s]
        y = p["true"].astype(int)
        assert np.array_equal(c["true"], y)
        ca, cb = c["pred"] == y, p["pred"] == y
        n01, n10 = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
        f1 = lambda yy, pp: precision_recall_fscore_support(yy, pp, average="macro", zero_division=0)[2]
        d_obs = f1(y, c["pred"]) - f1(y, p["pred"])
        boot = []
        for _ in range(10000):
            i = rng.integers(0, len(y), len(y))
            boot.append(f1(y[i], c["pred"][i]) - f1(y[i], p["pred"][i]))
        lo, hi = np.percentile(boot, [2.5, 97.5])
        out.append(dict(seed=s, delta_macro_f1_pp=100 * d_obs, ci95_low=100 * lo, ci95_high=100 * hi,
                        bootstrap_p=min(1, 2 * min(np.mean(np.array(boot) <= 0), np.mean(np.array(boot) >= 0))),
                        cafe_only_correct=n01, phobert_only_correct=n10,
                        mcnemar_exact_p=binomtest(n01, n01 + n10, 0.5).pvalue))
    if out:
        pd.DataFrame(out).to_csv(OUT / "G1_paired_tests.csv", index=False)
        log("G1 paired tests (CafeBERT - PhoBERT):\n" + pd.DataFrame(out).round(4).to_string(index=False))
    for f in sorted(OUT.glob("G*/metrics.json")):
        log(f"{f.parent.name}: {json.dumps(json.load(open(f))['results'])[:600]}")


for st in STAGES:
    log(f"===== {st}")
    {"G1": stage_g1, "G2": stage_g2, "G3": stage_g3, "SUMMARY": stage_summary}[st]()
log("Done. Send the folder " + str(OUT) + ".")

## Dùng lại kết quả cũ trên Drive

In [ ]:
# Dùng lại kết quả của lần chạy trước trên Drive (nếu có) để bỏ qua các bước đã xong
import shutil, glob, os
for d in ["revision_data", "revision_data/preds", "revision_results"]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{RESULT_DIR}/{d}/*"):
        if os.path.isfile(f) and not os.path.exists(f"{d}/{os.path.basename(f)}"):
            shutil.copy(f, f"{d}/")
have = lambda p: os.path.exists(p)
print("preds:", len(glob.glob("revision_data/preds/*.npz")), "| corpus_index:", have("revision_data/corpus_index.pkl"),
      "| step2:", have("revision_results/step2_report.txt"))

## Chạy

In [ ]:
os.environ["ONLY_MODELS"] = "PhoBERT"
os.environ["STEP4_OUT"] = RESULT_DIR + "/step4"
# os.environ["PHOBERT_LR"] = "5e-5"               # mặc định 2e-5 (theo trainer_state đã công bố)
!nvidia-smi
if len(glob.glob("revision_data/preds/PhoBERT_seed*.npz")) < 3:
    !python step1_collect.py
if not have("revision_data/corpus_index.pkl"):
    !python step1b_index.py
!python step4_gpu.py G3

## Lưu kết quả

In [ ]:
# Chép kết quả sang Drive và tạo file zip để tải về
import shutil, glob
for d in ["revision_results", "revision_data"]:
    for f in glob.glob(f"{d}/*.txt") + glob.glob(f"{d}/*.csv") + glob.glob(f"{d}/*.tex") + \
             glob.glob(f"{d}/*.pdf") + glob.glob(f"{d}/*.tif") + glob.glob(f"{d}/*.pkl"):
        if os.path.getsize(f) < 500e6:
            os.makedirs(f"{RESULT_DIR}/{d}", exist_ok=True)
            shutil.copy(f, f"{RESULT_DIR}/{d}/")
os.makedirs(f"{RESULT_DIR}/revision_data/preds", exist_ok=True)
for f in glob.glob("revision_data/preds/*.npz"):
    shutil.copy(f, f"{RESULT_DIR}/revision_data/preds/")
print("Đã chép vào", RESULT_DIR)